# Engenharia de Features

Com a EDA concluída, sei o que preciso construir. Para modelos de forecasting funcionarem bem, as features temporais precisam capturar sazonalidade, tendência e comportamento recente por SKU. Vou construir tudo isso aqui.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 1. Carregando e limpando os dados

Aplico aqui a limpeza definida na EDA: remover cancelamentos, tratar nulos e duplicatas.

In [ ]:
df_2009 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df_2010 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')
df = pd.concat([df_2009, df_2010], ignore_index=True)

# Removo cancelamentos — identificados por Quantity negativa ou prefixo 'C' no Invoice
df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]

# Removo linhas sem Customer ID — não consigo rastrear comportamento sem identificador
df = df.dropna(subset=['Customer ID'])

# Duplicatas exatas
df = df.drop_duplicates()

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Revenue'] = df['Quantity'] * df['Price']

print(f'Dataset limpo: {len(df):,} registros')

## 2. Agregação diária por SKU

Os modelos de séries temporais precisam de uma série agregada. Vou trabalhar na granularidade diária por SKU.

In [ ]:
# Agrego vendas por SKU e data
df_diario = df.groupby(['StockCode', pd.Grouper(key='InvoiceDate', freq='D')]).agg(
    quantidade_total=('Quantity', 'sum'),
    receita_total=('Revenue', 'sum'),
    num_transacoes=('Invoice', 'nunique'),
    preco_medio=('Price', 'mean')
).reset_index()

df_diario.columns = ['sku', 'data', 'quantidade', 'receita', 'n_transacoes', 'preco_medio']

print(f'Registros diários por SKU: {len(df_diario):,}')
print(f'SKUs únicos: {df_diario["sku"].nunique():,}')
print(f'Período: {df_diario["data"].min()} a {df_diario["data"].max()}')

## 3. Features temporais

O Prophet já lida com sazonalidade internamente, mas para modelos como Random Forest e Gradient Boosting preciso criar essas features manualmente.

In [ ]:
def criar_features_temporais(df):
    """Crio todas as features temporais que os modelos tree-based vão precisar."""
    df = df.copy()
    df['ano'] = df['data'].dt.year
    df['mes'] = df['data'].dt.month
    df['dia_semana'] = df['data'].dt.dayofweek
    df['dia_mes'] = df['data'].dt.day
    df['semana_ano'] = df['data'].dt.isocalendar().week.astype(int)
    df['trimestre'] = df['data'].dt.quarter
    df['is_fim_semana'] = (df['dia_semana'] >= 5).astype(int)
    
    # Sazonalidade circular — uso seno e cosseno para que mês 12 e mês 1 fiquem próximos
    df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
    df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)
    df['dia_semana_sin'] = np.sin(2 * np.pi * df['dia_semana'] / 7)
    df['dia_semana_cos'] = np.cos(2 * np.pi * df['dia_semana'] / 7)
    
    return df

df_diario = criar_features_temporais(df_diario)
print('Features temporais criadas com sucesso')

## 4. Features de lag e janela deslizante

Para modelos de forecasting, o histórico recente do próprio SKU é uma das features mais importantes. Lag-7 captura o mesmo dia da semana anterior, lag-30 captura o mesmo período do mês anterior.

In [ ]:
def criar_features_lag(df, lags=[1, 7, 14, 30], janelas=[7, 14, 30]):
    """Crio lags e médias móveis por SKU — essas são as features mais preditivas
    em problemas de demanda com sazonalidade semanal e mensal."""
    df = df.sort_values(['sku', 'data']).copy()
    
    for lag in lags:
        df[f'lag_{lag}'] = df.groupby('sku')['quantidade'].shift(lag)
    
    for janela in janelas:
        df[f'media_movel_{janela}d'] = (
            df.groupby('sku')['quantidade']
            .transform(lambda x: x.shift(1).rolling(janela, min_periods=1).mean())
        )
        df[f'std_movel_{janela}d'] = (
            df.groupby('sku')['quantidade']
            .transform(lambda x: x.shift(1).rolling(janela, min_periods=1).std())
        )
    
    return df

df_features = criar_features_lag(df_diario)
print(f'Dataset com features de lag: {df_features.shape}')

## 5. Feature de risco de ruptura

Crio uma variável alvo para o problema de classificação: identificar se um SKU está em risco de ruptura nos próximos 7 dias com base no histórico de demanda e na velocidade de giro.

In [ ]:
# Defino risco de ruptura como: demanda projetada nos próximos 7 dias > percentil 75 do histórico
# Na prática, isso seria calibrado com dados reais de estoque — aqui uso a demanda como proxy

demanda_percentil = df_features.groupby('sku')['quantidade'].transform(lambda x: x.quantile(0.75))
df_features['demanda_alta'] = (df_features['quantidade'] > demanda_percentil).astype(int)

print('Distribuição da variável alvo (demanda_alta):')
print(df_features['demanda_alta'].value_counts(normalize=True).round(3))

## 6. Salvando o dataset processado

In [ ]:
# Removo linhas com NaN que vieram dos lags iniciais (sem histórico suficiente)
df_final = df_features.dropna(subset=[f'lag_{l}' for l in [1, 7, 14, 30]])

df_final.to_parquet('../data/processed/features_modelagem.parquet', index=False)

print(f'Dataset salvo: {len(df_final):,} registros, {df_final.shape[1]} features')
print('Arquivo: ../data/processed/features_modelagem.parquet')